In [1]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START,END
from langgraph.types import Send

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

In [12]:
class Task(BaseModel):
    id:int
    title: str 

    goal: str= Field(
        ...,
        description="one sentence describing what the reader should be able to do/understand after this section.",
    )
    bullets: List[str]=Field(
        ...,
        min_lenght=3,
        max_lenght=5,
        description="3-5 concrete, non-overlapping subpoints to cover in this section",
    )
    target_words: int= Field(
        ...,
        description="Target word count for this section (120-450).",
    )
    tags: List[str]=Field(default_factory=list)
    requires_research:bool=False
    requires_citations:bool=False
    requires_code:bool=False
    section_type: Literal["intro","core","examples","checklist","common_mistakes","conclusion"
                          ]=Field(
                              ...,
                              description="Use 'common_mistakes' exactly once in the plan.",
                          )
    
    brief: str= Field(...,description="What to cover")

/var/folders/yl/58rrbn4n4_s01rm82mlwmgkw0000gn/T/ipykernel_75975/2817229370.py:9: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'min_lenght', 'max_lenght'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  bullets: List[str]=Field(


In [13]:
class Plan(BaseModel):
    blog_title:str
    audience: str
    tone:str
    blog_kind: Literal["explainer","tutorial","news_roundup"]="explainer"
    contstraints:List[str]=Field(default_factory=True)
    tasks: List[Task]

In [ ]:
class EvidenceItem(BaseModel):
    title:str
    url:str
    published_at: Optional[str]=None
    snippet: Optional[str]=None
    source:Optional[str]=None

In [15]:
class RouterDecision(BaseModel):
    needs_research:bool
    mode: Literal["closed_book","hybrid","open_book"]
    queries: List[str]=Field(default_factory=True)

In [16]:
class EvidencePack(BaseModel):
    evidence: List[EvidenceItem]=Field(default_factory=True)

In [19]:
class State(TypedDict):
    topic: str

    #comes from routing/search decision
    mode:str
    needs_research:bool
    queries:List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]


    #reducer: results from workers get concatenated automatically
    sections: Annotated[list[tuple[int,str]],operator.add] #{task_id,section_md}
    final:str

In [ ]:
#ROUTER_SYSTEM

In [5]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [6]:
def orchestrator(state:State)->dict:
    plan=llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=("Create a blog plan with 5-7 sections on the following topic.")
            ),
            HumanMessage(content=f"Topic:{state['topic']}"),
        ]  
    )
    return {"plan":plan}

In [7]:
#fanout to check how mnay tasks are their in plan object
def fanout(state:State):
    return [Send("worker",{"task":task,"topic":state["topic"],"plan":state["plan"]})
            for task in state["plan"].tasks]


In [8]:
def worker(payload:dict)->dict:

    #payload contains what we sent
    task=payload["task"]
    topic=payload["topic"]
    plan=payload["plan"]

    blog_title=plan.blog_title

    section_md=llm.invoke(
        [
            SystemMessage(content="Write one clean Markdown section"),
            HumanMessage(
                content=(
                    f"Blog:{blog_title}\n"
                    f"Audience :{plan.audience}\n"
                    f"Tone:{plan.tone}"
                    f"Topic:{topic}\n"
                    f"Section:{task.title}\n"
                    f"Section Type:{task.section_type}\n"
                    f"Goal:{task.goal}\n"
                    f"Target Words:{task.target_words}\n"
                    f"Bulets: {task.bullets}\n\n"
                    "Return only the section content in markdown."
                )
            )
        ]
    ).content.strip()

    return {"sections":[section_md]}




In [9]:
from pathlib import Path
def reducer(state:State)->dict:
    title=state["plan"].blog_title
    body="\n\n".join(state["sections"]).strip()

    final_md=f"# {title}\n\n{body}\n"

    #to save file
    filename=title.lower().replace(" ","_")+".md"
    output_path=Path(filename)
    output_path.write_text(final_md,encodign="utf-8")

    return {"final":final_md}


In [10]:
g=StateGraph(State)
g.add_node("orchestrator",orchestrator)
g.add_node("worker",worker)
g.add_node("reducer",reducer)


In [11]:
g.add_edge(START,"orchestrator")
g.add_conditional_edges("orchestrator", fanout)
g.add_edge("worker","reducer")
g.add_edge("reducer",END)

app=g.compile()

app

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
out=app.invoke({"topic":"Write a blog on self-attention"})

: 